# 06 Tuning

In [29]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from functools import partial

import lightgbm as lgbm
import xgboost as xgb

import sys
sys.path.append('../src')

from load_clean_data import load_clean_data
from backtest import get_cutoffs, run_backtest
from features import build_features
from evaluation import score_forecasts, merge_backtests
from models import add_lgbm_forecast, LGBM_PARAMS, add_xgb_forecast, XGB_PARAMS
from tuning import random_search

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [15]:
df = load_clean_data()
df_with_features = build_features(df)

In [16]:
cutoffs = get_cutoffs(df['Date'].max(), 7)
first_10 = cutoffs[:10]

In [17]:
XGB_SEARCH_SPACE = {'max_depth': ('int', 2, 10),
                    'learning_rate': ('log', 0.01, 0.3),
                    'n_estimators': ('int', 100, 600),
                    'min_child_weight': ('int', 1, 50),
                    'subsample': ('float', 0.5, 1.0),
                    'colsample_bytree': ('float', 0.5, 1.0),
                    'reg_lambda': ('log', 0.1, 10)
                    }

LGBM_SEARCH_SPACE = {'num_leaves': ('int', 4, 127),
                     'learning_rate': ('log', 0.01, 0.3),
                     'n_estimators': ('int', 100, 600),
                     'min_child_samples': ('int', 10, 200),
                     'subsample': ('float', 0.5, 1.0),
                     'colsample_bytree': ('float', 0.5, 1.0),
                     'reg_lambda': ('log', 0.1, 10)
                    }

In [18]:
RUN_XGB_SEARCH = False

In [19]:
if RUN_XGB_SEARCH:
    xgb_tuning = random_search(df_with_features, first_10, add_xgb_forecast, 'xgb_forecast', XGB_PARAMS, XGB_SEARCH_SPACE, n_trials=25, path='../results/tuning/xgb_tuning.csv')

    print(xgb_tuning.head(5))

else:
    xgb_tuning = pd.read_csv('../results/tuning/xgb_tuning.csv')

    print(xgb_tuning.head(5))

   trial  max_depth  learning_rate  n_estimators  min_child_weight  subsample  \
0     11          4       0.047629           491              26.0   0.569898   
1     15          5       0.066497           431              26.0   0.832157   
2     10          5       0.019737           503               1.0   0.893462   
3     17          8       0.054967           180              47.0   0.848160   
4      4          9       0.012424           530              42.0   0.815832   

   colsample_bytree  reg_lambda         RMSE         MAE       WAPE  \
0          0.557265    2.171730  1037.676514  600.600098  10.396508   
1          0.703193    4.246594  1046.446045  603.257812  10.442514   
2          0.832425    2.572354  1040.912842  604.042297  10.456093   
3          0.723078    0.578153  1053.123291  606.449585  10.497764   
4          0.879044    0.511743  1054.557495  608.962952  10.541272   

      seconds  
0  130.736621  
1  120.236105  
2  147.615089  
3   85.254588  
4  227

In [20]:
RUN_LGBM_SEARCH = False

In [21]:
if RUN_LGBM_SEARCH:
    lgbm_tuning = random_search(df_with_features, first_10, add_lgbm_forecast, 'lgbm_forecast', LGBM_PARAMS, LGBM_SEARCH_SPACE, n_trials=25, path='../results/tuning/lgbm_tuning.csv')

    print(lgbm_tuning.head(5))

else:
    lgbm_tuning = pd.read_csv('../results/tuning/lgbm_tuning.csv')

    print(lgbm_tuning.head(5))

   trial  num_leaves  learning_rate  n_estimators  min_child_samples  \
0     11          38       0.047629           491              106.0   
1     24          55       0.016349           439               29.0   
2      1          15       0.044491           487               92.0   
3     18          22       0.027885           442              130.0   
4     17          92       0.054967           180              189.0   

   subsample  colsample_bytree  reg_lambda         RMSE         MAE  \
0   0.569898          0.557265    2.171730  1041.569247  594.547667   
1   0.793822          0.585296    7.083375  1046.619979  601.214989   
2   0.848684          0.547089    8.938090  1046.770635  601.831498   
3   0.680906          0.543825    0.172192  1053.008182  603.261592   
4   0.848160          0.723078    0.578153  1055.978529  606.623891   

        WAPE    seconds  
0  10.291740  50.417771  
1  10.407152  62.228377  
2  10.417824  50.147137  
3  10.442579  48.768198  
4  10.5007

In [22]:
XGB_TUNED_PARAMS = {
    'tree_method': 'hist',
    'enable_categorical': True,
    'n_estimators': 491,
    'learning_rate': 0.047629,
    'max_depth': 4,
    'min_child_weight': 26,
    'subsample': 0.569898,
    'colsample_bytree': 0.557265,
    'reg_lambda': 2.17173,
    'random_state': 42
}

LGBM_TUNED_PARAMS = {
    'n_estimators': 491,
    'learning_rate': 0.047629,
    'num_leaves': 38,
    'subsample_freq': 1,
    'min_child_samples': 106,
    'subsample': 0.569898,
    'colsample_bytree': 0.557265,
    'reg_lambda': 2.17173,
    'random_state': 42,
    'verbose': -1
}

In [ ]:
FORECAST = False

if FORECAST:
    add_tuned_xgb = partial(add_xgb_forecast, params=XGB_TUNED_PARAMS)

    bt = run_backtest(df_with_features, cutoffs, 7, add_tuned_xgb)

    bt = bt.rename(columns={'xgb_forecast': 'xgb_tuned_forecast'})

    bt = bt[['Store', 'Date', 'Sales', 'cutoff', 'day_ahead', 'xgb_tuned_forecast']]

    bt.to_csv('../results/backtests/xgb_tuned_backtest.csv', index=False)

else: 
    bt = pd.read_csv('../results/backtests/xgb_tuned_backtest.csv', parse_dates=['Date', 'cutoff'], low_memory=False)

In [25]:
SAVE_SCORES = True

if SAVE_SCORES:
    scores = score_forecasts(bt, ['xgb_tuned_forecast'])
    scores.to_csv('../results/xgb_tuned_scores.csv', index=False)

In [27]:
FORECAST = True

if FORECAST:
    add_tuned_lgbm = partial(add_lgbm_forecast, params=LGBM_TUNED_PARAMS)

    bt = run_backtest(df_with_features, cutoffs, 7, add_tuned_lgbm)

    bt = bt.rename(columns={'lgbm_forecast': 'lgbm_tuned_forecast'})

    bt = bt[['Store', 'Date', 'Sales', 'cutoff', 'day_ahead', 'lgbm_tuned_forecast']]

    bt.to_csv('../results/backtests/lgbm_tuned_backtest.csv', index=False)

else: 
    bt = pd.read_csv('../results/backtests/lgbm_tuned_backtest.csv', parse_dates=['Date', 'cutoff'], low_memory=False)

In [28]:
SAVE_SCORES = True

if SAVE_SCORES:
    scores = score_forecasts(bt, ['lgbm_tuned_forecast'])
    scores.to_csv('../results/lgbm_tuned_scores.csv', index=False)

In [31]:
combined_bt = merge_backtests(['baselines', 'xgb', 'lgbm', 'xgb_tuned', 'lgbm_tuned'])

forecasts = ['naive_baseline', 'avg_28d_baseline', 'avg_all_baseline', 'xgb_forecast', 'xgb_tuned_forecast', 'lgbm_forecast', 'lgbm_tuned_forecast']

combined_scores = score_forecasts(combined_bt, forecasts)

model_names = {
    'naive_baseline': 'Naive Baseline',
    'avg_28d_baseline': '28-Day Average Baseline',
    'avg_all_baseline': 'Full History Average Baseline',
    'xgb_forecast': 'XGBoost Forecast',
    'xgb_tuned_forecast': 'XGBoost Tuned',
    'lgbm_forecast': 'LGBM Forecast',
    'lgbm_tuned_forecast': 'LGBM Tuned'
}

relabelled_scores = combined_scores.copy()

relabelled_scores['Model'] = relabelled_scores['Model'].replace(model_names)

relabelled_scores.sort_values(by='WAPE').style.hide(axis='index').format({ 'WAPE': '{:.2f}%' }, precision=2)

Model,RMSE,MAE,WAPE
LGBM Tuned,994.02,596.66,10.18%
XGBoost Tuned,991.48,604.18,10.31%
XGBoost Forecast,1001.30,604.95,10.33%
LGBM Forecast,1035.76,619.47,10.57%
Full History Average Baseline,2029.48,1261.49,21.53%
28-Day Average Baseline,2127.55,1326.10,22.63%
Naive Baseline,3206.68,2112.59,36.06%
